# Aula 02 — Sanitização amostral por Chauvenet

Nesta aula, vamos ler a saída da Aula 1 e aplicar o critério de Chauvenet.

## Objetivos

- consumir a amostra unitarizada;
- aplicar a sanitização;
- analisar o histórico de iterações;
- salvar a amostra saneada para a Aula 3.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import resolve_project_root
from servicos.sanitizacao import build_sanitization_report

In [2]:
def locate_input_file(project_root: Path) -> Path:
    candidate = project_root / 'data' / 'output' / 'aula_01_amostra_unitarizada.csv'
    if candidate.exists():
        return candidate
    raise FileNotFoundError('Saída da Aula 1 não encontrada em data/output/.')

## Etapa 1 — Ler a saída da Aula 1

In [3]:
project_root = resolve_project_root()
input_path = locate_input_file(project_root)
df_unitized = pd.read_csv(input_path)
print('Base carregada da Aula 1:', df_unitized.shape)
df_unitized.head()

Base carregada da Aula 1: (20, 8)


,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink,valor_unitario
0,AP-001,750000.0,85.0,2,5.0,1.2,https://portalimoveis.com.br/anuncio/001,8823.529412
1,AP-002,820000.0,92.5,2,8.0,1.8,https://portalimoveis.com.br/anuncio/002,8864.864865
2,AP-003,690000.0,78.0,1,12.0,2.5,https://portalimoveis.com.br/anuncio/003,8846.153846
3,AP-004,1200000.0,115.0,3,2.0,0.8,https://portalimoveis.com.br/anuncio/004,10434.782609
4,AP-005,580000.0,65.0,1,20.0,4.2,https://portalimoveis.com.br/anuncio/005,8923.076923


## Etapa 2 — Aplicar Chauvenet

In [4]:
report = build_sanitization_report(df_unitized)
df_clean = report['df_saneado']
df_removed = report['df_removidos']
history_df = report['historico_iteracoes']
print('Amostra inicial:', report['amostra_inicial'])
print('Amostra saneada:', report['amostra_saneada'])
print('Total removido:', report['total_removido'])

Amostra inicial: 20
Amostra saneada: 17
Total removido: 3


## Etapa 3 — Histórico e removidos

In [5]:
history_df

,iteracao,n_inicial,media,desvio_padrao,probabilidade_limite,z_critico,removidos,status
0,1,20,9334.191143,2771.918277,0.025000,2.241403,1,outliers_removidos
1,2,19,8772.832782,1207.390854,0.026316,2.221520,1,outliers_removidos
2,3,18,9037.990159,359.311725,0.027778,2.200411,1,outliers_removidos
3,4,17,8955.825897,89.787249,0.029412,2.177923,0,sem_outliers


In [6]:
if isinstance(df_removed, pd.DataFrame) and not df_removed.empty:
    display(df_removed)
else:
    print('Nenhuma observação foi removida pelo critério de Chauvenet.')

,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink,valor_unitario,z_score,chauvenet_outlier,z_critical,probability_limit,iteracao_remocao
0,AP-009,2400000.0,120.0,3,1.0,0.5,https://portalimoveis.com.br/anuncio/009,20000.000000,3.847808,True,2.241403,0.025000,1
1,AP-014,320000.0,80.0,1,25.0,5.0,https://portalimoveis.com.br/anuncio/014,4000.000000,3.953014,True,2.221520,0.026316,2
2,AP-004,1200000.0,115.0,3,2.0,0.8,https://portalimoveis.com.br/anuncio/004,10434.782609,3.887411,True,2.200411,0.027778,3


## Etapa 4 — Salvar saída para a Aula 3

In [7]:
output_dir = project_root / 'data' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'aula_02_amostra_saneada.csv'
df_clean.to_csv(output_path, index=False)
print(output_path)

/Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia/data/output/aula_02_amostra_saneada.csv


## Conclusão

A Aula 2 termina com a amostra saneada salva em arquivo. A Aula 3 vai consumir essa saída.